In [1]:
import cv2
import os

user_id = input("请输入用户ID（数字）: ")
cam = cv2.VideoCapture(0, cv2.CAP_V4L2)  # 避免 GStreamer 警告

# 确保 haarcascade 文件路径正确
face_detector = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")

count = 0
while True:
    ret, img = cam.read()
    if not ret:
        print("无法读取摄像头画面")
        break

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_detector.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        count += 1
        cv2.imwrite(f"dataset/User.{user_id}.{count}.jpg", gray[y:y+h, x:x+w])
        cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2)

    cv2.imshow('image', img)

    if cv2.waitKey(100) & 0xFF == ord('q'):
        break
    elif count >= 30:
        break

cam.release()
cv2.destroyAllWindows()

无法读取摄像头画面


In [4]:
import cv2
import numpy as np
from os import listdir
from os.path import isfile, join

data_path = 'dataset/'
training_data, labels = [], []

onlyfiles = [f for f in listdir(data_path) if isfile(join(data_path, f))]

for file in onlyfiles:
    image_path = data_path + file
    
    # 假设文件名格式是 User.{id}.{num}.jpg
    parts = file.split('.')
    if len(parts) < 4:
        print(f"跳过非法文件名: {file}")
        continue

    try:
        label = int(parts[1])  # 第二个部分是用户ID
    except ValueError:
        print(f"无法解析标签: {file}")
        continue

    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    training_data.append(np.asarray(image, dtype=np.uint8))
    labels.append(label)

labels = np.array(labels, dtype=np.int32)

model = cv2.face.LBPHFaceRecognizer_create()
model.train(training_data, labels)
model.save('trained_model.yml')
print("模型训练完成并保存为 trained_model.yml")

AttributeError: module 'cv2' has no attribute 'face'

In [6]:
import cv2
import numpy as np
from os import listdir
from os.path import isfile, join

data_path = 'dataset/'
training_data, labels = [], []

onlyfiles = [f for f in listdir(data_path) if isfile(join(data_path, f))]

for file in onlyfiles:
    image_path = data_path + file
    
    # 假设文件名格式是 User.{id}.{num}.jpg
    parts = file.split('.')
    if len(parts) < 4:
        print(f"跳过非法文件名: {file}")
        continue

    try:
        label = int(parts[1])  # 第二个部分是用户ID
    except ValueError:
        print(f"无法解析标签: {file}")
        continue

    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    training_data.append(np.asarray(image, dtype=np.uint8))
    labels.append(label)

labels = np.array(labels, dtype=np.int32)

model = cv2.face.LBPHFaceRecognizer_create()
model.train(training_data, labels)
model.save('trained_model.yml')
print("模型训练完成并保存为 trained_model.yml")

模型训练完成并保存为 trained_model.yml


In [ ]:
import cv2

# 加载预训练模型
model = cv2.face.LBPHFaceRecognizer_create()
model.read('trained_model.yml')

# 加载人脸检测分类器
face_detector = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")

cap = cv2.VideoCapture(0)
font = cv2.FONT_HERSHEY_SIMPLEX

while True:
    ret, frame = cap.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(100, 100))

    for (x, y, w, h) in faces:
        face = gray[y:y+h, x:x+w]
        label, confidence = model.predict(face)

        # 显示结果
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(frame, f"ID: {label} ({confidence:.2f})", (x, y - 10), font, 0.7, (0, 255, 0), 2)

    cv2.imshow('Face Recognition', frame)

    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


(python:2934): GStreamer-WARNING **: 10:26:39.481: External plugin loader failed. This most likely means that the plugin loader helper binary was not found or could not be run. You might need to set the GST_PLUGIN_SCANNER environment variable if your setup is unusual. This should normally not be required though.
[ WARN:0@1099.195] global cap_gstreamer.cpp:1173 isPipelinePlaying OpenCV | GStreamer warning: GStreamer: pipeline have not been created
